In [11]:


import numpy as np
import pandas as pd
import os

np.random.seed(42)

rows = 1000

data = {
    "Pregnancies": np.random.randint(0, 11, rows),
    "Glucose": np.random.randint(70, 200, rows),
    "BloodPressure": np.random.randint(60, 100, rows),
    "SkinThickness": np.random.randint(10, 50, rows),
    "Insulin": np.random.randint(15, 300, rows),
    "BMI": np.round(np.random.uniform(18.0, 45.0, rows), 1),
    "Age": np.random.randint(21, 70, rows)
}

df = pd.DataFrame(data)
df.insert(0,"Patient_ID",range(1,rows+1))

df["Glucose"] += np.random.normal(0, 5, rows)
df["BMI"] += np.random.normal(0, 1, rows)

def calculate_outcome(row):
    risk_score = 0

    if row["Glucose"] > 140:
        risk_score += 2
    if row["BMI"] > 35:
        risk_score += 1
    if row["Age"] > 50:
        risk_score += 1
    if row["BloodPressure"] > 85:
        risk_score += 1
    if row["Insulin"] > 180:
        risk_score += 1

    return 1 if risk_score >= 3 else 0

df["Outcome"] = df.apply(calculate_outcome, axis=1)

for col in df.columns:
    if col !="Patient_ID":
        df.loc[df.sample(frac=0.02).index, col] = np.nan

df.fillna(df.median(numeric_only=True), inplace=True)

os.makedirs("data", exist_ok=True)

file_path = "data/diabetes_dataset_1000_rows.csv"
df.to_csv(file_path, index=False)

print(df)
print("\nDataset Shape:", df.shape)

print("\nClass Distribution:")
print(df["Outcome"].value_counts())
df.to_csv("dbrisk1.csv",index=False)

     Patient_ID  Pregnancies     Glucose  BloodPressure  SkinThickness  \
0             1          6.0  102.814670           70.0           10.0   
1             2          3.0  176.326845           90.0           24.0   
2             3         10.0   79.019316           74.0           46.0   
3             4          7.0   96.249683           68.0           22.0   
4             5          4.0  186.781979           82.0           49.0   
..          ...          ...         ...            ...            ...   
995         996          9.0   97.506831           62.0           25.0   
996         997          5.0   69.244078           60.0           17.0   
997         998          1.0   77.290891           95.0           49.0   
998         999          7.0  152.965363           84.0           49.0   
999        1000          7.0  199.239127           96.0           11.0   

     Insulin        BMI   Age  Outcome  
0      108.0  32.335804  30.0      0.0  
1      236.0  32.605962  30.0

In [17]:


import pandas as pd

df = pd.read_csv("dbrisk1.csv")
pd.set_option('display.max_column',None)
pd.set_option('display.width',1000)
pd.set_option('display.max_colwidth',None)
#preprocessing
print(df.head())
print(df.info())


#for checking missing values
print(df.isnull().sum())


#for detection of outliers& removal
Q1= df.quantile(0.25)
Q3= df.quantile(0.75)
IQR = Q3 - Q1
df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]
print("After removing outliers:", df.shape)

#feature and target split
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

#feature scaling 
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#feature selection

from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(score_func=chi2, k=5)
X_new = selector.fit_transform(abs(X), y)
selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features)





 

   Patient_ID  Pregnancies     Glucose  BloodPressure  SkinThickness  Insulin        BMI  Age  Outcome
0           1            6  102.814670             70             10    108.0  32.335804   30        0
1           2            3  176.326845             90             24    236.0  32.605962   30        1
2           3           10   79.019316             74             46    218.0  42.755146   58        1
3           4            7   96.249683             68             22    152.0  33.102327   47        0
4           5            4  186.781979             82             49    221.0  26.031863   41        1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Patient_ID     1000 non-null   int64  
 1   Pregnancies    1000 non-null   int64  
 2   Glucose        1000 non-null   float64
 3   BloodPressure  1000 non-null   int64  
 4   SkinThic

In [ ]:
#split the dataset 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)